# Handwritten word → text with EmbeddingGemma 2

Zero-shot: embed an image of a word with the vision tower, embed a list of English words
with the text tower, and pick the word whose embedding is closest (cosine similarity).

In [ ]:
# On Colab or a fresh machine, uncomment to install:
# !pip install -q -U "transformers>=5.20" "sentence-transformers>=6.1" wordfreq pillow

In [ ]:
import os
os.environ["TQDM_DISABLE"] = "1"            # no progress bars (they can hang the notebook UI)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch, numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
from wordfreq import top_n_list

# use the GPU if there is one: CUDA (Colab/Linux), MPS (Apple Silicon), else CPU
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# text + vision only (skip the audio encoder), bf16 weights
model = SentenceTransformer("google/embeddinggemma-2", device=device,
                            config_kwargs={"audio_config": None},
                            model_kwargs={"dtype": torch.bfloat16})

## 1. Embed the vocabulary (once)

In [ ]:
words = [w for w in top_n_list("en", 30000) if w.isalpha()]   # 30k most common English words
word_emb = model.encode(words, prompt_name="query", normalize_embeddings=True, batch_size=256)
word_emb.shape

## 2. Embed a PNG and find the nearest word

In [ ]:
def read_word(png_path, topk=5):
    img = Image.open(png_path).convert("RGB")
    img_emb = model.encode({"image": img}, normalize_embeddings=True,
                           processing_kwargs={"image": {"max_soft_tokens": 70}})
    sims = word_emb @ img_emb                      # cosine similarity (both unit length)
    best = np.argsort(-sims)[:topk]
    return [(words[i], round(float(sims[i]), 3)) for i in best]

read_word("galaxy.png")

In [ ]:
Image.open("galaxy.png")